# LangGraph + Azure AI Search RAG Example

This notebook is a LangGraph-based version of the original Semantic Kernel Azure AI Search (RAG) example. It demonstrates:

- Initializing Azure AI Search as a simple retrieval store
- Defining retrieval and utility tools (documents + weather info)
- Creating a tool-calling agent using LangChain + Azure OpenAI
- Streaming responses and displaying function calls/results

Environment variables expected (set in a .env file):
- `AZURE_AI_FOUNDRY_ENDPOINT` (Azure OpenAI endpoint)
- `AZURE_OPENAI_API_VERSION` (API version)
- `AZURE_AI_FOUNDRY_MODEL` (Deployment name / model id)
- `AZURE_AI_FOUNDRY_API_KEY` (Key)
- `AZURE_SEARCH_SERVICE_ENDPOINT` (Azure AI Search endpoint)
- `AZURE_SEARCH_API_KEY` (Azure AI Search admin/query key)

> Note: If the index already exists it will be reused. Documents are uploaded each run for simplicity.

## Install Dependencies

In [ ]:
%pip install --quiet python-dotenv azure-search-documents langchain langgraph langchain-openai openai

## Imports & Environment Setup

In [5]:
import os, json
from dotenv import load_dotenv
from typing import List

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchableField, SearchFieldDataType

from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

from langchain.agents import AgentExecutor, create_tool_calling_agent
from IPython.display import display, HTML

load_dotenv()
azure_endpoint = os.getenv('AZURE_AI_FOUNDRY_ENDPOINT')
api_version = os.getenv('AZURE_OPENAI_API_VERSION')
model_name = os.getenv('AZURE_AI_FOUNDRY_MODEL')
api_key = os.getenv('AZURE_AI_FOUNDRY_API_KEY')
search_service_endpoint = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT')
search_api_key = os.getenv('AZURE_SEARCH_API_KEY')

print('Azure Endpoint:', azure_endpoint)
print('Model Name:', model_name)
print('Search Endpoint:', search_service_endpoint)

if not all([azure_endpoint, api_version, model_name, api_key, search_service_endpoint, search_api_key]):
    raise ValueError('One or more required environment variables are missing. Please check your .env file.')

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Adobe\ai-agents\.venv\Lib\site-packages\langchain\agents\__init__.py)

## Azure AI Search Index Initialization

In [ ]:
index_name = 'travel-documents'
search_client = SearchClient(endpoint=search_service_endpoint, index_name=index_name, credential=AzureKeyCredential(search_api_key))
index_client = SearchIndexClient(endpoint=search_service_endpoint, credential=AzureKeyCredential(search_api_key))

fields = [
    SimpleField(name='id', type=SearchFieldDataType.String, key=True),
    SearchableField(name='content', type=SearchFieldDataType.String)
 ]
index = SearchIndex(name=index_name, fields=fields)

try:
    existing_index = index_client.get_index(index_name)
    print(f"Index '{index_name}' exists. Reusing.")
except Exception:
    print(f"Creating index '{index_name}' ...")
    index_client.create_index(index)

documents = [
    {'id': '1', 'content': 'Contoso Travel offers luxury vacation packages to exotic destinations worldwide.'},
    {'id': '2', 'content': 'Our premium travel services include personalized itinerary planning and 24/7 concierge support.'},
    {'id': '3', 'content': "Contoso's travel insurance covers medical emergencies, trip cancellations, and lost baggage."},
    {'id': '4', 'content': 'Popular destinations include the Maldives, Swiss Alps, and African safaris.'},
    {'id': '5', 'content': 'Contoso Travel provides exclusive access to boutique hotels and private guided tours.'}
 ]
search_client.upload_documents(documents)
print('Uploaded/updated sample documents.')

## Define Retrieval and Utility Tools

In [ ]:
@tool
def retrieve_documents(query: str) -> str:
    """Retrieve matching documents from Azure AI Search and return concatenated context."""
    results = search_client.search(query)
    context_parts: List[str] = []
    for r in results:  # streaming iterator
        try:
            context_parts.append(f"Document: {r['content']}")
        except KeyError:
            pass
    return '\n\n'.join(context_parts) if context_parts else 'No results found.'

@tool
def get_destination_temperature(destination: str) -> str:
    """Return average temperature for known destinations."""
    temps = {
        'maldives': '82°F (28°C)',
        'swiss alps': '45°F (7°C)',
        'african safaris': '75°F (24°C)',
    }
    key = destination.lower()
    if key in temps:
        return f"The average temperature in {destination} is {temps[key]}."
    return (
        "Sorry, I don't have temperature information for {destination}. "
        "Available: Maldives, Swiss Alps, African safaris.")

tools = [retrieve_documents, get_destination_temperature]
print('Tools registered:', [t.name for t in tools])

## Create Tool-Calling Agent (LangChain)

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    openai_api_version=api_version,
    deployment_name=model_name,
    api_key=api_key,
    temperature=0.2,
)

system_prompt = (
    "You are TravelAgent, a helpful travel assistant. Use tools when needed. "
    "If retrieval context is available from retrieve_documents, summarize it first. "
    "If a temperature is requested, call get_destination_temperature. "
    "If information is not found in context, say so clearly but still try relevant tools."
 )
prompt = ChatPromptTemplate.from_messages([('system', system_prompt), ('human', '{input}')])
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=False, stream_runnable=True)
print('Agent and executor ready.')

## Run Streaming Demo

In [ ]:
def stream_agent_responses(user_inputs):
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )
        function_calls = []
        final_text_parts = []
        for event in executor.stream({'input': user_input}):
            if 'tool_calls' in event:
                for call in event['tool_calls']:
                    fn_name = call.get('name')
                    args = call.get('args')
                    function_calls.append(f"Calling tool: {fn_name}({json.dumps(args)})")
            if 'output' in event and isinstance(event['output'], str):
                final_text_parts.append(event['output'])
        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Tool Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )
        html_output += (
            "<div style='margin-bottom:20px'>"
            "<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap>{''.join(final_text_parts)}</div></div><hr>"
        )
        display(HTML(html_output))

sample_questions = [
    "Can you explain Contoso's travel insurance coverage?",
    "What is the average temperature of the Maldives?",
    "What is a good cold destination offered by Contoso and what is its average temperature?",
]
stream_agent_responses(sample_questions)

## Expected Sample Output
You should see user queries followed by expandable tool call details and the TravelAgent's streamed responses summarizing retrieval context and tool results.